# core

> Fill in a module description here

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

: 

## Method map for Linked Data Knowledge
```python
class LinkedDataKnowledge:
    def __init__(self, data=None)
    def __repr__(self)
    def _repr_markdown_(self)
    def find_entity(self, entity_id=None, term_type=None, label=None)
    def _has_type(self, entity, type_str)
    def get_entity_description(self, entity)
    def query(self, query_type, query_value)
    def describe(self, path="")
    def view(self, entity_id=None, term_type=None, label=None)
```

Helper functions:
```python
def jsonld_merge(docs)
```

Tests: 
```python
#| test
def test_initialization():
    kb = LinkedDataKnowledge()
    test_eq(kb.data, {"@context": {}, "@graph": []})
    
    test_data = {"@context": {"schema": "https://schema.org/"}, "@graph": [{"@id": "test"}]}
    kb2 = LinkedDataKnowledge(test_data)
    test_eq(kb2.data, test_data)

#| test
def test_find_entity():
    test_kb = LinkedDataKnowledge({
        "@context": {},
        "@graph": [
            {
                "@id": "http://example.org/Person",
                "@type": "rdfs:Class",
                "rdfs:label": "Person"
            },
            {
                "@id": "http://example.org/name",
                "@type": "rdf:Property",
                "rdfs:label": "name"
            }
        ]
    })

    # Test find by ID
    person_results = test_kb.find_entity(entity_id="Person")
    test_eq(len(person_results), 1)
    test_eq(person_results[0]['@id'], "http://example.org/Person")

    # Test find by type
    class_results = test_kb.find_entity(term_type="rdfs:Class")
    test_eq(len(class_results), 1)
    test_eq(class_results[0]['@id'], "http://example.org/Person")

    # Test find by label
    label_results = test_kb.find_entity(label="Person")
    test_eq(len(label_results), 1)
    test_eq(label_results[0]['@id'], "http://example.org/Person")

#| test
def test_get_entity_description():
    test_kb = LinkedDataKnowledge()
    test_entity = {
        "@id": "http://example.org/Person",
        "@type": "rdfs:Class",
        "rdfs:label": "Person",
        "rdfs:comment": "A person (alive, dead, undead, or fictional).",
        "rdfs:subClassOf": {"@id": "http://schema.org/Thing"}
    }

    description = test_kb.get_entity_description(test_entity)
    test_assert("Entity: http://example.org/Person" in description)
    test_assert("Type: rdfs:Class" in description)
    test_assert("Labels: Person" in description)
    test_assert("A person (alive, dead, undead, or fictional)" in description)
    test_assert("rdfs:subClassOf: http://schema.org/Thing" in description)

#| test
def test_markdown_representation():
    kb = LinkedDataKnowledge({
        "@context": {"schema": "https://schema.org/"},
        "@graph": [
            {
                "@id": "http://example.org/person1",
                "@type": "schema:Person",
                "schema:name": "Alice Smith"
            }
        ]
    })
    
    md = kb._repr_markdown_()
    test_assert('LinkedDataKnowledge' in md)
    test_assert('Context' in md)
    test_assert('Graph' in md)
    test_assert('schema:Person' in md)
```

In [ ]:
#| exports

from fastcore.basics import *
from fastcore.meta import *
from fastcore.test import *
import json
from rdflib import Graph
from pyld import jsonld
from typing import List, Dict, Any, Optional, Union

In [ ]:
#| export
class LinkedDataKnowledge:
    "Represents a knowledge base of linked data in JSON-LD format"
    def __init__(self, 
                 data:Dict=None, # Initial knowledge data
                ):
        self.data = data or {"@context": {}, "@graph": []}
        
    def __repr__(self): return f"LinkedDataKnowledge with {len(self.data.get('@graph', []))} entities"

In [ ]:
#| export
@patch
def _repr_markdown_(self:LinkedDataKnowledge) -> str:
    "Rich markdown representation for notebook display"
    md = [f"## LinkedDataKnowledge"]
    
    # Context summary
    context = self.data.get('@context', {})
    md.append(f"### Context ({len(context)} prefixes)")
    
    # Show the first few context entries
    if context:
        md.append("```json")
        context_preview = dict(list(context.items())[:5])
        if len(context) > 5:
            md.append(json.dumps(context_preview, indent=2) + "\n... and more")
        else:
            md.append(json.dumps(context, indent=2))
        md.append("```")
    
    # Graph summary
    graph = self.data.get('@graph', [])
    md.append(f"### Graph ({len(graph)} entities)")
    
    # Show types of entities in the graph
    if graph:
        types = {}
        for entity in graph:
            entity_type = entity.get('@type')
            if isinstance(entity_type, list):
                for t in entity_type:
                    types[t] = types.get(t, 0) + 1
            elif entity_type:
                types[entity_type] = types.get(entity_type, 0) + 1
        
        if types:
            md.append("**Entity types:**")
            for t, count in sorted(types.items(), key=lambda x: x[1], reverse=True):
                md.append(f"- {t}: {count}")
    
    # Show a sample entity if available
    if graph:
        md.append("\n**Sample entity:**")
        md.append("```json")
        md.append(json.dumps(graph[0], indent=2))
        md.append("```")
    
    # If using @included, show that too
    if '@included' in self.data:
        included = self.data['@included']
        md.append(f"\n### Included ({len(included)} entities)")
        md.append("**Types of included entities:**")
        
        # Count types of included entities
        included_types = {}
        for entity in included:
            entity_type = entity.get('@type')
            if isinstance(entity_type, list):
                for t in entity_type:
                    included_types[t] = included_types.get(t, 0) + 1
            elif entity_type:
                included_types[entity_type] = included_types.get(entity_type, 0) + 1
        
        for t, count in sorted(included_types.items(), key=lambda x: x[1], reverse=True):
            md.append(f"- {t}: {count}")
    
    return "\n".join(md)


In [ ]:
#| export
def _format_entity_markdown(entity:Dict) -> str:
    "Format a single entity as markdown"
    md = []
    
    # Entity ID and type
    entity_id = entity.get('@id', 'No ID')
    entity_type = entity.get('@type', ['Unknown'])
    if isinstance(entity_type, list):
        entity_type = ', '.join(entity_type)
    
    md.append(f"### {entity_type}: {entity_id}")
    
    # Properties
    for prop, values in entity.items():
        if prop in ['@id', '@type']:
            continue
            
        # Format the property name
        prop_name = prop.split('/')[-1] if '/' in prop else prop
        md.append(f"**{prop_name}**:")
        
        # Format values
        for val in values:
            if isinstance(val, dict):
                if '@value' in val:
                    value_text = val['@value']
                    if '@language' in val:
                        value_text += f" @{val['@language']}"
                    md.append(f"- {value_text}")
                elif '@id' in val:
                    md.append(f"- [{val['@id']}]({val['@id']})")
                else:
                    md.append(f"- {val}")
            else:
                md.append(f"- {val}")
    
    return "\n".join(md)

@patch
def query_markdown(self:LinkedDataKnowledge,
                  query_type:str, # Type of query: "property", "type", "value" 
                  query_value:str, # Value to search for
                  ) -> str:
    "Query the knowledge base and return results as markdown"
    results = self.query(query_type, query_value)
    
    if not results:
        return f"*No results found for {query_type}='{query_value}'*"
    
    md = [f"# Query Results: {query_type}='{query_value}'", 
          f"Found {len(results)} matching entities"]
    
    # Format each result entity
    for i, entity in enumerate(results[:5]):  # Limit to first 5 for readability
        md.append(f"\n## Result {i+1}")
        md.append(_format_entity_markdown(entity))
    
    if len(results) > 5:
        md.append(f"\n*...and {len(results)-5} more results*")
    
    return "\n".join(md)


In [ ]:
@patch
def find_entity(self:LinkedDataKnowledge, 
               entity_id:str=None, # Full or partial entity ID to find
               term_type:str=None, # Filter by type (e.g., "Class", "Property")
               label:str=None # Find by label text
              ) -> list: # Matching entities
    "Find entities in the graph by ID, type, or label"
    results = []
    graph = self.data.get('@graph', [])
    
    for entity in graph:
        # Match by ID if specified
        if entity_id and isinstance(entity.get('@id'), str):
            if entity_id in entity['@id']:
                if term_type is None or self._has_type(entity, term_type):
                    results.append(entity)
                    continue
        
        # Match by label if specified
        if label and not entity_id:
            for key, value in entity.items():
                if 'label' in key.lower():
                    if isinstance(value, list):
                        for item in value:
                            if isinstance(item, dict) and item.get('@value') == label:
                                if term_type is None or self._has_type(entity, term_type):
                                    results.append(entity)
                                    break
                    elif isinstance(value, str) and label in value:
                        if term_type is None or self._has_type(entity, term_type):
                            results.append(entity)
                            break
    
    return results

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()